# Colab 31 — SNNEED training-size vs. build-cost (all 4 evaluation sets)

Train the *same* SNNEED (encoder + 3-bin classifier head — the colab29b architecture, verbatim) **from
scratch** at `N ∈ {10k, 30k, 100k}` training pairs, `EPOCHS = 30`, 3 seeds each, and for every run measure
both **quality** and the **wall-clock cost to build it**.

**Evaluation sets (4):** `synth` (disjoint synthetic set, built with the *training* generator), `3Di`, `SS`,
`AA`. The encoder is **AA-trained** and scored zero-shot on SS/3Di (the transfer story). Metrics per set:
**Spearman** ρ(sim, normLev), **AUROC** (≥0.70), and **MAP@10** (full-pool retrieval).

**Deliverables**
1. A **final table**: 4 sets × {Spearman, AUROC, MAP@10} at the full budget (N=100k), plus the per-N version.
2. **Retrieval (MAP@10) vs training size** — all 4 sets as curves in one graph.
3. **Quality vs build-cost** — MAP@10 against the seconds it took to train, with a one-line efficiency verdict.

**Notes.** No CSV is ever read (a receipt CSV is *written* at the end). AUROC here is on the **stratified pair
set** (same pairs as Spearman) — cheap and internally consistent, *not* the full-pool colab29b AUROC, and the
fresh training means numbers won't match the run-of-record exactly. **Honesty line:** epochs fixed at 30, so
larger N also means more gradient steps — the realistic *"same schedule, more data"* curve.

## 1. Setup (self-contained — clones the repo for the CATH data)

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
for f in ['cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz', 'cath_s20_3di.csv.gz']:
    p = os.path.join(DATA_DIR, f); print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch rapidfuzz scikit-learn scipy matplotlib --quiet

In [ ]:
import time, numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from rapidfuzz.distance import Levenshtein as RFLev
from rapidfuzz.process import cdist as rf_cdist
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# ---- config ----
N_GRID = [10_000, 30_000, 100_000]   # training-pair budgets to compare (100_000 == 100k)
SEEDS  = [0, 1, 2]                    # set to [0] for a ~3x faster single-seed pass
EPOCHS = 30                          # loss converges well by 30 (final CE printed per run)
STRAT_PER_BIN = 400
STRAT_CAND    = 200_000
SYN_PERTURB, SYN_INDEP = 20_000, 8_000   # synthetic eval feed (disjoint from training set)

FEED_ORDER = ['synth', '3Di', 'SS', 'AA']            # display order (matches the deck)
FEED_COLOR = {'synth': '#ff7f0e', '3Di': '#1f77b4', 'SS': '#d62728', 'AA': '#7f7f7f'}

## 2. SNNEED architecture (encoder + classifier head — colab29b, verbatim) + timed trainer

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'; SS_ALPHABET = 'HLS'
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}; PAD_IDX = 20; VOCAB = 21
MIN_LEN, MAX_LEN, BS, K = 50, 200, 128, 16
BAND_LOW_AA, BAND_HIGH = 0.30, 0.70
AA_SET, SS_SET = set(AA_ALPHABET), set(SS_ALPHABET)
is_aa = lambda s: all(c in AA_SET for c in s); is_ss = lambda s: all(c in SS_SET for c in s)
def norm_lev(a, b):
    L = max(len(a), len(b)); return 1.0 if L == 0 else 1.0 - RFLev.distance(a, b) / L
def encode_pad(seq):
    idx = [CHAR_TO_IDX[c] for c in seq][:MAX_LEN]; idx += [PAD_IDX]*(MAX_LEN-len(idx))
    return torch.tensor(idx, dtype=torch.long)
def perturb(seq, k, abc, rng):
    s = list(seq); abc = list(abc)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= MAX_LEN: op = rng.choice(['sub', 'del'])
        else: op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub': i = rng.integers(0, len(s)); s[i] = rng.choice([c for c in abc if c != s[i]])
        elif op == 'ins': i = rng.integers(0, len(s)+1); s.insert(i, rng.choice(abc))
        else: i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)
def rand_seq(abc, rng): L = int(rng.integers(MIN_LEN, MAX_LEN+1)); return ''.join(rng.choice(list(abc), size=L))
def bin_idx(x, band_low): return 0 if x < band_low else (1 if x < BAND_HIGH else 2)

class Enc(nn.Module):   # SNNEED encoder (colab29b)
    def __init__(s):
        super().__init__(); s.emb = nn.Embedding(VOCAB, 32, padding_idx=PAD_IDX)
        s.c1 = nn.Conv1d(32, 32, 3, padding=1); s.c2 = nn.Conv1d(32, 64, 3, padding=1)
        s.pool = nn.AdaptiveAvgPool1d(K); s.fc = nn.Linear(64*K, 128)
    def forward(s, x):
        m = (x != PAD_IDX).float(); e = s.emb(x).permute(0, 2, 1)
        h = F.relu(s.c1(e)); h = F.relu(s.c2(h)); h = h * m.unsqueeze(1)
        return F.normalize(s.fc(s.pool(h).flatten(1)), p=2, dim=1)
class Clf(nn.Module):   # encoder + 3-bin classifier head (colab29b)
    def __init__(s):
        super().__init__(); s.encoder = Enc()
        s.head = nn.Sequential(nn.Linear(128, 64), nn.LeakyReLU(0.01), nn.Linear(64, 3))
    def forward(s, a, b): return s.head(torch.abs(s.encoder(a) - s.encoder(b)))
class DS(Dataset):
    def __init__(s, pp, bl): s.p = pp; s.bl = bl
    def __len__(s): return len(s.p)
    def __getitem__(s, i): a, b, l = s.p[i]; return encode_pad(a), encode_pad(b), bin_idx(l, s.bl)

def train_snn_n(alphabet, band_low, n_train, seed, label):
    """colab29b.train_snn recipe with N + seed explicit; returns (model, datagen_s, train_s)."""
    rng = np.random.default_rng(seed); torch.manual_seed(seed)
    t0 = time.perf_counter(); pairs = []
    while len(pairs) < n_train:
        sd = rand_seq(alphabet, rng); L = len(sd); t = float(rng.uniform(0, 1)); k = max(0, int(round((1-t)*L)))
        o = perturb(sd, k, alphabet, rng)
        if 1 <= len(o) <= MAX_LEN: pairs.append((sd, o, norm_lev(sd, o)))
    datagen_s = time.perf_counter() - t0
    dl = DataLoader(DS(pairs, band_low), batch_size=BS, shuffle=True)
    model = Clf().to(device); opt = torch.optim.Adam(model.parameters(), 1e-3); crit = nn.CrossEntropyLoss()
    model.train(); t0 = time.perf_counter()
    for ep in range(1, EPOCHS+1):
        tot = nb = 0
        for a, b, y in dl:
            a, b, y = a.to(device), b.to(device), y.to(device)
            loss = crit(model(a, b), y); opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item(); nb += 1
        if ep % 10 == 0 or ep == 1: print(f'  [{label}] epoch {ep}/{EPOCHS} CE {tot/nb:.4f}')
    if device.type == 'cuda': torch.cuda.synchronize()
    train_s = time.perf_counter() - t0
    model.eval(); return model, datagen_s, train_s

snn_params = sum(p.numel() for p in Enc().parameters())
print(f'SNNEED encoder params = {snn_params:,}')

## 3. Real pools (AA/SS/3Di) + oracles + stratified pairs (built once, shared)

In [ ]:
raw = pd.concat([pd.read_csv(f'{DATA_DIR}/cath_s20_train70.csv.gz'),
                 pd.read_csv(f'{DATA_DIR}/cath_s20_test30.csv.gz')],
                ignore_index=True).drop_duplicates('domain_id')
seqs3 = pd.read_csv(f'{DATA_DIR}/cath_s20_3di.csv.gz')
RESCUED = {'4z0mC02', '3qkaE02'}
def _valid(seq, isstd, d):
    return (isinstance(seq, str) and isstd(seq) and ((MIN_LEN <= len(seq) <= MAX_LEN) or d in RESCUED))
id_to_aa  = {d: s for d, s in zip(raw['domain_id'], raw['aa_seq'])              if _valid(s, is_aa, d)}
id_to_ss  = {d: s for d, s in zip(raw['domain_id'], raw['ss_seq'])              if _valid(s, is_ss, d)}
id_to_3di = {d: s for d, s in zip(seqs3['domain_id'], seqs3['3di'].astype(str)) if _valid(s, is_aa, d)}
LOOK = {'AA': id_to_aa, 'SS': id_to_ss, '3Di': id_to_3di}
POOL_SEQ = {f: list(LOOK[f].values()) for f in LOOK}
CATH_FEEDS = ['AA', 'SS', '3Di']
for f in CATH_FEEDS: print(f'  {f:<4} pool = {len(POOL_SEQ[f]):>6}')

In [ ]:
# --- exhaustive de-hubbed oracle: T_high[q] = indices with normLev>=0.70 (self excluded) ---
def build_oracle(feed, block=1024):
    seqs = POOL_SEQ[feed]; lens = np.array([len(s) for s in seqs]); N = len(seqs)
    T_high = {}; pos_pairs = []
    for r0 in range(0, N, block):
        r1 = min(r0 + block, N)
        Dm = rf_cdist(seqs[r0:r1], seqs, scorer=RFLev.distance, workers=-1).astype(np.float64)
        den = np.maximum(lens[r0:r1][:, None], lens[None, :]); den[den == 0] = 1
        sim = 1.0 - Dm / den
        for a in range(r1 - r0):
            i = r0 + a; row = sim[a].copy(); row[i] = -1.0
            hi = np.where(row >= BAND_HIGH)[0]
            if hi.size: T_high[i] = hi.astype(np.int32)
            for j in hi:
                if j > i: pos_pairs.append((i, int(j), float(row[j])))
    return dict(T_high=T_high, pos_pairs=pos_pairs)

ORACLE = {}
for f in CATH_FEEDS:                                  # SS is the heavy one (~10k queries) — a few minutes
    print(f'Building {f} oracle (exhaustive Levenshtein)...')
    ORACLE[f] = build_oracle(f)
    print(f'  {f}: queries@0.70 = {len(ORACLE[f]["T_high"]):>6}, high-sim pos pairs = {len(ORACLE[f]["pos_pairs"]):>7}')

In [ ]:
# --- stratified pair set per real feed (per-feed normLev deciles) — for Spearman + AUROC ---
def build_strat_pairs(feed, rng):
    seqs = POOL_SEQ[feed]; N = len(seqs)
    a = rng.integers(0, N, STRAT_CAND); b = rng.integers(0, N, STRAT_CAND)
    keep = a != b; a, b = a[keep], b[keep]
    nl = np.array([norm_lev(seqs[i], seqs[j]) for i, j in zip(a, b)])
    if feed in ORACLE and ORACLE[feed]['pos_pairs']:
        parr = np.array(ORACLE[feed]['pos_pairs'], dtype=float)
        a = np.concatenate([a, parr[:, 0].astype(np.int64)])
        b = np.concatenate([b, parr[:, 1].astype(np.int64)])
        nl = np.concatenate([nl, parr[:, 2]])
    edges = np.linspace(0.0, 1.0, 11); bins = np.clip(np.digitize(nl, edges) - 1, 0, 9)
    ai, aj, av = [], [], []
    for bb in range(10):
        idx = np.where(bins == bb)[0]
        if idx.size == 0: continue
        take = rng.permutation(idx)[:STRAT_PER_BIN]
        ai.append(a[take]); aj.append(b[take]); av.append(nl[take])
    return dict(i=np.concatenate(ai).astype(np.int64), j=np.concatenate(aj).astype(np.int64),
               nl=np.concatenate(av))
strat_rng = np.random.default_rng(999)
STRAT = {f: build_strat_pairs(f, strat_rng) for f in CATH_FEEDS}
for f in CATH_FEEDS:
    print(f'  strat[{f:<3}] = {len(STRAT[f]["nl"]):>5} pairs  (max normLev {STRAT[f]["nl"].max():.2f}, '
          f'{(STRAT[f]["nl"]>=BAND_HIGH).sum()} high-sim)')

In [ ]:
# --- synthetic eval feed: disjoint from training, built with the SAME generator, stratified ---
# perturbation pairs (mirror training) + independent random pairs (fill the low-sim deciles).
def build_synth_feed(n_perturb, n_indep, per_bin=STRAT_PER_BIN, seed=20260810):
    r = np.random.default_rng(seed); recs = []
    for _ in range(n_perturb):
        base = rand_seq(AA_ALPHABET, r); part = perturb(base, int(r.integers(0, len(base)+1)), AA_ALPHABET, r)
        if 1 <= len(part) <= MAX_LEN: recs.append((base, part))
    for _ in range(n_indep):
        recs.append((rand_seq(AA_ALPHABET, r), rand_seq(AA_ALPHABET, r)))
    recs = [(a, b, norm_lev(a, b)) for a, b in recs]
    nl_all = np.array([x[2] for x in recs]); bins = np.clip(np.digitize(nl_all, np.linspace(0, 1, 11)) - 1, 0, 9)
    take = []
    for bb in range(10):
        idx = np.where(bins == bb)[0]
        if idx.size: take.extend(r.permutation(idx)[:per_bin].tolist())
    seqs, I, J, NL = [], [], [], []
    for idx in take:
        a, b, nl = recs[int(idx)]
        I.append(len(seqs)); seqs.append(a); J.append(len(seqs)); seqs.append(b); NL.append(nl)
    return seqs, np.array(I), np.array(J), np.array(NL)

SYN_SEQ, SYN_I, SYN_J, SYN_NL = build_synth_feed(SYN_PERTURB, SYN_INDEP)
POOL_SEQ['synth'] = SYN_SEQ
print(f'Building synth oracle over {len(SYN_SEQ)} sequences...')
ORACLE['synth'] = build_oracle('synth')
print(f'  synth: pool={len(SYN_SEQ)}, pairs={len(SYN_NL)} (normLev {SYN_NL.min():.2f}-{SYN_NL.max():.2f}, '
      f'{(SYN_NL>=BAND_HIGH).sum()} high-sim), queries@0.70={len(ORACLE["synth"]["T_high"])}')

## 4. Metric helpers (embed → Spearman / AUROC / GPU-batched MAP@10)

In [ ]:
@torch.no_grad()
def embed_torch(model, seqs, bs=256):
    outs = []
    for i in range(0, len(seqs), bs):
        x = torch.stack([encode_pad(s) for s in seqs[i:i+bs]]).to(device)
        outs.append(model.encoder(x))
    return torch.cat(outs)                      # (N,128), L2-normalized, on device

def _auroc(sim, nl):
    y = (nl >= BAND_HIGH).astype(int)
    return roc_auc_score(y, sim) if 0 < y.sum() < len(y) else np.nan

def map10_torch(E, T_high, k=10, qb=256):
    q = list(T_high.keys())
    if not q: return np.nan
    aps = []
    for s0 in range(0, len(q), qb):
        qi = q[s0:s0+qb]
        sc = E[qi] @ E.t()                      # (b, N) cosine (E is unit-norm)
        for r, idx in enumerate(qi): sc[r, idx] = -1e9   # exclude self
        top = torch.topk(sc, k, dim=1).indices.cpu().numpy()
        for r, idx in enumerate(qi):
            ts = set(T_high[idx].tolist()); hits = 0; ap = 0.0
            for rr, o in enumerate(top[r], 1):
                if o in ts: hits += 1; ap += hits / rr
            aps.append(ap / min(len(ts), k))
    return float(np.mean(aps))

def eval_feed(model, feed):
    E = embed_torch(model, POOL_SEQ[feed]); Enp = E.cpu().numpy()
    if feed == 'synth':
        sim = np.sum(Enp[SYN_I] * Enp[SYN_J], axis=1); nl = SYN_NL
    else:
        P = STRAT[feed]; sim = np.sum(Enp[P['i']] * Enp[P['j']], axis=1); nl = P['nl']
    sp = spearmanr(sim, nl).correlation
    au = _auroc(sim, nl)
    mp = map10_torch(E, ORACLE[feed]['T_high'])
    return sp, au, mp

## 5. Train at each N — record quality (4 feeds) **and** build cost

In [ ]:
res_rows, cost_rows = [], []
wall0 = time.perf_counter()
for N in N_GRID:
    for seed in SEEDS:
        model, datagen_s, train_s = train_snn_n(AA_ALPHABET, BAND_LOW_AA, N, seed, f'N={N},s={seed}')
        cost_rows.append(dict(N=N, seed=seed, datagen_s=datagen_s, train_s=train_s, build_s=datagen_s+train_s))
        line = []
        for feed in FEED_ORDER:
            sp, au, mp = eval_feed(model, feed)
            res_rows.append(dict(N=N, seed=seed, feed=feed, spearman=sp, auroc=au, map10=mp))
            line.append(f'{feed}: ρ={sp:.2f} AUROC={au:.2f} MAP={mp:.2f}')
        print(f'  N={N:>7} seed={seed}  build={cost_rows[-1]["build_s"]:.1f}s  |  ' + '  '.join(line))
res  = pd.DataFrame(res_rows)
cost = pd.DataFrame(cost_rows)
res.to_csv('colab31_metrics.csv', index=False)          # write-only receipts
cost.to_csv('colab31_cost.csv', index=False)
print(f'\nTotal wall-clock for all runs: {time.perf_counter()-wall0:.1f}s')

## 6. Final table — all 4 sets × {Spearman, AUROC, MAP@10}

In [ ]:
N_TABLE = max(N_GRID)
def table_at(N):
    t = (res[res.N == N].groupby('feed')[['spearman', 'auroc', 'map10']]
         .mean().reindex(FEED_ORDER))
    t.columns = ['Spearman', 'AUROC(>=0.70)', 'MAP@10']
    return t.round(3)

print('=' * 60)
print(f'SNNEED — final capability table  (N={N_TABLE:,}, mean over {len(SEEDS)} seeds)')
print('=' * 60)
print(table_at(N_TABLE).to_string())

print('\n' + '=' * 60); print('Same table at every training budget'); print('=' * 60)
full = (res.groupby(['N', 'feed'])[['spearman', 'auroc', 'map10']].mean()
        .reindex(pd.MultiIndex.from_product([N_GRID, FEED_ORDER], names=['N', 'feed'])).round(3))
print(full.to_string())
table_at(N_TABLE).to_csv('colab31_final_table.csv')

## 7. Figure A — Spearman | AUROC | MAP@10 vs training size (all 4 sets)

In [ ]:
import matplotlib.pyplot as plt
def despine(ax):
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

xpos = np.arange(len(N_GRID))                     # categorical -> evenly spaced, axis starts at origin
xlabels = [f'{n//1000}k' for n in N_GRID]
def stat(feed, col):
    g = res[res.feed == feed].groupby('N')[col]
    return g.mean().reindex(N_GRID).values, g.std().reindex(N_GRID).fillna(0).values

fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
for a, col, title in [(ax[0], 'spearman', 'Spearman ρ(sim, normLev)'),
                      (ax[1], 'auroc', 'AUROC (≥0.70, stratified)'),
                      (ax[2], 'map10', 'MAP@10 (full-pool retrieval)')]:
    for feed in FEED_ORDER:
        m, sd = stat(feed, col)
        a.plot(xpos, m, 'o-', color=FEED_COLOR[feed], label=feed)
        a.fill_between(xpos, m - sd, m + sd, color=FEED_COLOR[feed], alpha=0.15)
    a.set_xticks(xpos); a.set_xticklabels(xlabels); a.set_xlim(-0.2, len(N_GRID) - 0.8)
    a.set_xlabel('training pairs N'); a.set_title(title); despine(a)
ax[0].legend(title='eval set', fontsize=9)
plt.tight_layout(); plt.savefig('colab31_metrics_vs_N.png', dpi=150, bbox_inches='tight'); plt.show()

## 8. Figure B — retrieval (MAP@10) vs build cost, all 4 sets + efficiency verdict

In [ ]:
tcost = cost.groupby('N')['build_s'].mean().reindex(N_GRID).values
tstd  = cost.groupby('N')['build_s'].std().reindex(N_GRID).fillna(0).values

fig, ax = plt.subplots(figsize=(8.4, 5.2))
for feed in FEED_ORDER:
    m, sd = stat(feed, 'map10')
    ax.errorbar(tcost, m, xerr=tstd, yerr=sd, fmt='o-', color=FEED_COLOR[feed], label=feed, capsize=3)
for k, n in enumerate(N_GRID):                     # annotate the budget next to each x position
    ax.annotate(f'{n//1000}k', (tcost[k], ax.get_ylim()[0]), textcoords='offset points',
                xytext=(0, 4), ha='center', fontsize=9, color='#555')
ax.set_xlim(left=0); ax.set_xlabel('build time (s)  [datagen + training, mean over seeds]')
ax.set_ylabel('MAP@10 (full-pool)'); ax.set_title('Retrieval quality vs. cost to build the encoder')
ax.legend(title='eval set'); despine(ax)
plt.tight_layout(); plt.savefig('colab31_map_vs_cost.png', dpi=150, bbox_inches='tight'); plt.show()

# --- efficiency verdict: is the top budget worth it over the next one down? ---
base, top = N_GRID[-2], N_GRID[-1]
tb = cost[cost.N == base]['build_s'].mean(); tt = cost[cost.N == top]['build_s'].mean()
print(f'Build time: N={base//1000}k = {tb:.0f}s vs N={top//1000}k = {tt:.0f}s  ({tb/tt:.0%} of the cost).')
for feed in FEED_ORDER:
    qb = res[(res.N == base) & (res.feed == feed)]['map10'].mean()
    qt = res[(res.N == top)  & (res.feed == feed)]['map10'].mean()
    print(f'  {feed:<5} MAP@10: N={base//1000}k reaches {qb/qt:.0%} of N={top//1000}k ({qb:.3f} vs {qt:.3f}).')

## 9. How to read this

- **Final table (§6)** — the headline capability numbers per evaluation set at the full budget, with the
  per-N version underneath so you can see what each extra doubling of data actually bought.
- **Figure A (§7)** — Spearman, AUROC, and MAP@10 vs N, one curve per set (synth orange, 3Di blue, SS red,
  AA grey). Read `synth` as the in-distribution reference and SS/3Di as the zero-shot transfer; **real-AA is
  the noise-level control** (only ~5–6 natural high-sim AA pairs, so its Spearman/MAP are jumpy — do not
  over-read them).
- **Figure B (§8)** — the same retrieval quality plotted against the **seconds to build** each encoder, with
  the efficiency verdict printed: what fraction of the top budget's MAP@10 the next budget down reaches, and
  at what fraction of the cost. That is the one-line defense of the chosen training size.

**Caveats.** AUROC is on the stratified pair set (not the full-pool colab29b AUROC). Epochs fixed at 30, so
larger N also means more gradient steps ("same schedule, more data", not compute-controlled). To probe
epochs, lower `EPOCHS` and watch the printed final CE (already well-converged by 30).

Outputs (written, not read): `colab31_metrics.csv`, `colab31_cost.csv`, `colab31_final_table.csv`,
`colab31_metrics_vs_N.png`, `colab31_map_vs_cost.png`.